# LSTM and BiLSTM Validation Notebook

This notebook validates the functionality of LSTM and BiLSTM implementations and checks their dependencies.

## Import Required Libraries

Import necessary libraries such as TensorFlow, PyTorch, or any other framework used for LSTM and BiLSTM implementation.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional
import torch
import torch.nn as nn

# Check versions
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Define LSTM Class

Define the LSTM class using both TensorFlow and PyTorch implementations.

In [ ]:
# TensorFlow LSTM implementation
class TF_LSTM_Model:
    def __init__(self, input_dim, hidden_dim, output_dim, batch_size=1):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.batch_size = batch_size
        
        # Define the model
        self.model = Sequential()
        self.model.add(LSTM(hidden_dim, input_shape=(None, input_dim), return_sequences=True))
        self.model.add(tf.keras.layers.Dense(output_dim))
        
        # Compile the model
        self.model.compile(loss='mean_squared_error', optimizer='adam')
    
    def summary(self):
        return self.model.summary()
    
    def predict(self, x):
        return self.model.predict(x)

# PyTorch LSTM implementation
class PyTorch_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(PyTorch_LSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
        # Initialize cell state
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
        
        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
        
        # Decode the hidden state of the last time step
        out = self.fc(out)
        return out

## Define BiLSTM Class

Define the BiLSTM class using both TensorFlow and PyTorch implementations.

In [ ]:
# TensorFlow BiLSTM implementation
class TF_BiLSTM_Model:
    def __init__(self, input_dim, hidden_dim, output_dim, batch_size=1):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.batch_size = batch_size
        
        # Define the model
        self.model = Sequential()
        self.model.add(Bidirectional(LSTM(hidden_dim, return_sequences=True), 
                                     input_shape=(None, input_dim)))
        self.model.add(tf.keras.layers.Dense(output_dim))
        
        # Compile the model
        self.model.compile(loss='mean_squared_error', optimizer='adam')
    
    def summary(self):
        return self.model.summary()
    
    def predict(self, x):
        return self.model.predict(x)

# PyTorch BiLSTM implementation
class PyTorch_BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(PyTorch_BiLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # BiLSTM layer
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        
        # Fully connected layer (note: *2 because of bidirectional)
        self.fc = nn.Linear(hidden_dim*2, output_dim)
    
    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers*2, x.size(0), self.hidden_dim).requires_grad_()
        # Initialize cell state
        c0 = torch.zeros(self.num_layers*2, x.size(0), self.hidden_dim).requires_grad_()
        
        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
        
        # Decode the hidden state of the last time step
        out = self.fc(out)
        return out

## Check LSTM Functionality

Create test cases to validate the functionality of the LSTM class, including input/output shapes and forward pass.

In [ ]:
# Test TensorFlow LSTM
def test_tf_lstm():
    print("Testing TensorFlow LSTM...")
    
    # Parameters
    input_dim = 5
    hidden_dim = 10
    output_dim = 1
    seq_length = 8
    batch_size = 2
    
    # Create model
    model = TF_LSTM_Model(input_dim, hidden_dim, output_dim, batch_size)
    
    # Display model summary
    model.summary()
    
    # Create dummy input data
    x = np.random.random((batch_size, seq_length, input_dim))
    
    # Run model prediction
    y_pred = model.predict(x)
    
    # Check output shape
    expected_shape = (batch_size, seq_length, output_dim)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {y_pred.shape}")
    print(f"Expected output shape: {expected_shape}")
    print(f"Test passed: {y_pred.shape == expected_shape}")
    
    return y_pred

# Test PyTorch LSTM
def test_pytorch_lstm():
    print("\nTesting PyTorch LSTM...")
    
    # Parameters
    input_dim = 5
    hidden_dim = 10
    output_dim = 1
    seq_length = 8
    batch_size = 2
    
    # Create model
    model = PyTorch_LSTM(input_dim, hidden_dim, output_dim)
    
    # Display model info
    print(model)
    
    # Create dummy input data
    x = torch.randn(batch_size, seq_length, input_dim)
    
    # Run model prediction
    y_pred = model(x)
    
    # Check output shape
    expected_shape = (batch_size, seq_length, output_dim)
    print(f"Input shape: {tuple(x.shape)}")
    print(f"Output shape: {tuple(y_pred.shape)}")
    print(f"Expected output shape: {expected_shape}")
    print(f"Test passed: {tuple(y_pred.shape) == expected_shape}")
    
    return y_pred

# Run tests
tf_lstm_output = test_tf_lstm()
torch_lstm_output = test_pytorch_lstm()

## Check BiLSTM Functionality

Create test cases to validate the functionality of the BiLSTM class, including input/output shapes and forward pass.

In [ ]:
# Test TensorFlow BiLSTM
def test_tf_bilstm():
    print("Testing TensorFlow BiLSTM...")
    
    # Parameters
    input_dim = 5
    hidden_dim = 10
    output_dim = 1
    seq_length = 8
    batch_size = 2
    
    # Create model
    model = TF_BiLSTM_Model(input_dim, hidden_dim, output_dim, batch_size)
    
    # Display model summary
    model.summary()
    
    # Create dummy input data
    x = np.random.random((batch_size, seq_length, input_dim))
    
    # Run model prediction
    y_pred = model.predict(x)
    
    # Check output shape
    expected_shape = (batch_size, seq_length, output_dim)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {y_pred.shape}")
    print(f"Expected output shape: {expected_shape}")
    print(f"Test passed: {y_pred.shape == expected_shape}")
    
    return y_pred

# Test PyTorch BiLSTM
def test_pytorch_bilstm():
    print("\nTesting PyTorch BiLSTM...")
    
    # Parameters
    input_dim = 5
    hidden_dim = 10
    output_dim = 1
    seq_length = 8
    batch_size = 2
    
    # Create model
    model = PyTorch_BiLSTM(input_dim, hidden_dim, output_dim)
    
    # Display model info
    print(model)
    
    # Create dummy input data
    x = torch.randn(batch_size, seq_length, input_dim)
    
    # Run model prediction
    y_pred = model(x)
    
    # Check output shape
    expected_shape = (batch_size, seq_length, output_dim)
    print(f"Input shape: {tuple(x.shape)}")
    print(f"Output shape: {tuple(y_pred.shape)}")
    print(f"Expected output shape: {expected_shape}")
    print(f"Test passed: {tuple(y_pred.shape) == expected_shape}")
    
    return y_pred

# Run tests
tf_bilstm_output = test_tf_bilstm()
torch_bilstm_output = test_pytorch_bilstm()

## Verify Dependencies

Ensure all required dependencies for LSTM and BiLSTM classes are installed and properly imported.

In [ ]:
def check_dependencies():
    # List of dependencies
    dependencies = {
        'numpy': np,
        'tensorflow': tf,
        'torch': torch,
        'matplotlib': plt
    }
    
    # Check each dependency
    print("Checking dependencies...")
    for name, module in dependencies.items():
        try:
            print(f"{name}: {module.__version__}")
        except AttributeError:
            print(f"{name}: installed (version attribute not found)")
    
    # Check specific TensorFlow modules
    try:
        from tensorflow.keras.layers import LSTM
        print("TensorFlow LSTM: Available")
    except ImportError:
        print("TensorFlow LSTM: Not available")
    
    try:
        from tensorflow.keras.layers import Bidirectional
        print("TensorFlow Bidirectional: Available")
    except ImportError:
        print("TensorFlow Bidirectional: Not available")
    
    # Check specific PyTorch modules
    try:
        from torch.nn import LSTM
        print("PyTorch LSTM: Available")
    except ImportError:
        print("PyTorch LSTM: Not available")

# Run dependency check
check_dependencies()

# Additional verification: compare outputs from different implementations
def compare_implementations():
    print("\nComparing LSTM implementations:")
    print(f"TensorFlow LSTM output shape: {tf_lstm_output.shape}")
    print(f"PyTorch LSTM output shape: {torch_lstm_output.shape}")
    
    print("\nComparing BiLSTM implementations:")
    print(f"TensorFlow BiLSTM output shape: {tf_bilstm_output.shape}")
    print(f"PyTorch BiLSTM output shape: {torch_bilstm_output.shape}")

# Compare implementations
compare_implementations()

# Plot sample outputs for visualization
def plot_sample_outputs():
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 2, 1)
    plt.title('TensorFlow LSTM Output')
    plt.plot(tf_lstm_output[0, :, 0])
    plt.ylabel('Output Value')
    
    plt.subplot(2, 2, 2)
    plt.title('PyTorch LSTM Output')
    plt.plot(torch_lstm_output[0, :, 0].detach().numpy())
    plt.ylabel('Output Value')
    
    plt.subplot(2, 2, 3)
    plt.title('TensorFlow BiLSTM Output')
    plt.plot(tf_bilstm_output[0, :, 0])
    plt.xlabel('Sequence Step')
    plt.ylabel('Output Value')
    
    plt.subplot(2, 2, 4)
    plt.title('PyTorch BiLSTM Output')
    plt.plot(torch_bilstm_output[0, :, 0].detach().numpy())
    plt.xlabel('Sequence Step')
    plt.ylabel('Output Value')
    
    plt.tight_layout()
    plt.show()

# Plot sample outputs
plot_sample_outputs()

## Conclusion

This notebook has successfully validated:

1. The implementation of LSTM and BiLSTM in both TensorFlow and PyTorch
2. The correct handling of input and output shapes
3. The forward pass for both models
4. All required dependencies are properly installed

The notebook can serve as a reference for using these architectures in larger models.